# CUDA 编程模型：从 CPU 思维切换到 GPU 思维

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-07-cuda-programming-model.ipynb)

在写第一个 kernel 之前，先把 CUDA 的世界观讲清楚。CPU 与 GPU 为什么不同、kernel / thread / block / grid 是什么、__global__ 和 <<< >>> 的含义、nvcc 怎样把 .cu 编译成 PTX 与 SASS、Compute Capability 7.5 对 T4 意味着什么，并在 Colab 里用 nvcc 编译运行第一个 CUDA C++ 程序。

本 notebook 配合教程章节 [CUDA 编程模型：从 CPU 思维切换到 GPU 思维](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

前六个 Lab 你一直在“用”GPU：PyTorch 替你分配显存、启动 kernel、同步结果。但软件层的算力配额（例如 HAMi 的 `gpucores`）是通过拦截 CUDA API 来限制 kernel 发射节奏实现的，MPS 让多进程的 kernel 并发在 SM 上，MIG 是把 SM 和显存物理切块：**这些机制全都建立在 CUDA 编程模型之上**。如果你不知道 kernel 是什么、block 怎么落到 SM、`<<<>>>` 里两个数字在干什么，后面的所有“共享”和“切分”讨论都只能停留在词汇层面。

本章的材料参考了 [moezdil/CUDA](https://github.com/moezdil/CUDA) 这套“先理解 GPU、再写 CUDA”的分步教程，把它压缩成一个 Colab 可以在 30 分钟内跑完的版本。

## CPU 与 GPU：为什么把代码搬到 GPU 并不自动变快

![CPU 与 GPU 的设计目标对比：CPU 用少数复杂核心与大缓存优化延迟，GPU 用 40 个 SM、2560 个简单核心优化吞吐](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/cpu-vs-gpu.svg)



| 维度 | CPU | GPU（T4） |
| :--- | :--- | :--- |
| 核心数量 | 几个到几十个 | 2560 个 CUDA Core，分布在 40 个 SM 上 |
| 单核心复杂度 | 高：乱序执行、分支预测、大缓存 | 低：简单 ALU，依赖并行度隐藏延迟 |
| 擅长 | 串行逻辑、复杂控制流、低延迟 | 同一段代码在海量数据上重复执行 |
| 调度单位 | 线程 | Warp（32 个线程锁步执行） |



关键结论：**GPU 快，是因为你给了它足够多可以同时做的事**。一个只启动 1 个线程的 kernel 在 T4 上比在 CPU 上还慢：本章第 2 步就是这样一个 kernel，它只证明“代码能在 GPU 上跑”，不代表快。

## CUDA 的四个核心概念



| 概念 | 含义 | 在代码里的样子 |
| :--- | :--- | :--- |
| **Kernel** | 在 GPU 上运行的函数，一次 launch 会被成千上万个线程同时执行 | `__global__ void add(...)` |
| **Thread** | 执行 kernel 的最小单元，每个线程有自己的编号 | `threadIdx.x` |
| **Block** | 一组线程，整体被派发到**一个** SM 上，内部可通过 shared memory 协作 | `blockIdx.x`、`blockDim.x` |
| **Grid** | 一次 launch 的全部 block | `gridDim.x` |



`__global__` 告诉 nvcc：这个函数由 CPU 调用、在 GPU 执行。另外两个限定符是 `__device__`（GPU 内部调用）和 `__host__`（普通 CPU 函数）。

启动语法 `kernel<<<blocks, threads>>>(args)` 里的两个数字分别是 block 数和每个 block 的线程数，总线程数是两者乘积。每个线程通过 `blockIdx.x * blockDim.x + threadIdx.x` 算出自己在全局的位置，这正是下一章 Numba 里 `cuda.grid(1)` 替你做的事。

> **kernel launch 是异步的**
>
> `kernel<<<...>>>()` 一执行，CPU 立刻往下走，不等 GPU。如果 `main()` 在 GPU 打印之前就返回了，你什么都看不到，且**不会有任何报错**。所以 launch 之后必须 `cudaDeviceSynchronize()`。这是新手遇到的第一个“编译通过、运行无错、输出为空”的问题。

## nvcc 做了什么：从 .cu 到 PTX 再到 SASS

![nvcc 把 .cu 拆成主机代码（交给 gcc/clang）和设备代码（先编译为 PTX 再编译为 sm_75 的 SASS），两者打包进同一个 fatbinary，运行时由驱动挑选 SASS 或即时编译 PTX](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/nvcc-pipeline.svg)

`nvcc` 不是一个普通编译器，它是一个**驱动程序**：把 `.cu` 文件拆成两部分，主机代码交给系统 C++ 编译器，设备代码走 NVIDIA 自己的两段流水线：

1. **PTX**（Parallel Thread Execution）：与具体 GPU 无关的虚拟指令集，类似“GPU 的汇编中间语言”。
2. **SASS**：某一代 GPU 的真实机器码。`-arch=sm_75` 就是让 `ptxas` 生成 Turing 架构的 SASS。

可执行文件里可以同时带 SASS 和 PTX。运行时驱动若找到匹配当前 GPU 的 SASS 就直接执行；若只有 PTX，则在启动时 JIT 编译成 SASS（启动变慢，但程序还能跑）。这就是为什么 PyTorch 的 wheel 包能同时支持多代 GPU。

## Compute Capability：T4 的 7.5 意味着什么

Compute Capability（CC）是 **GPU 硬件能力的版本号**，不是性能分数。主版本号代表架构换代，次版本号代表同代内的扩展：



| 架构 | CC | 代表产品 | 关键能力 |
| :--- | :--- | :--- | :--- |
| Volta | 7.0 | V100 | 首次引入 Tensor Core |
| **Turing** | **7.5** | **T4**、RTX 20 系 | Tensor Core 支持 INT8/INT4，**CUDA 13 支持的最老架构** |
| Ampere | 8.0 / 8.6 | A100 / RTX 30 系 | 第三代 Tensor Core、**MIG**（A100/A30） |
| Ada Lovelace | 8.9 | L4、L40S、RTX 40 系 | FP8 Tensor Core |
| Hopper | 9.0 | H100 / H200 | Transformer Engine、更大 NVLink |
| Blackwell | 10.x / 12.x | B200、RTX 50 系 | 第五代 Tensor Core、NVFP4 |



CC 决定两件事：**硬件有没有某个单元**（T4 没有 FP8 Tensor Core，再怎么写代码也用不上），以及**哪些 CUDA Toolkit 版本还支持你**（CUDA 13 已经放弃 Pascal / Volta，Turing 7.5 是底线）。所以写 CUDA 之前的第一问永远是“我的 GPU 支持什么”，然后才是“它够不够快”。

## CUDA 平台栈：你在 Colab 里免费拿到的全部

当你在 Colab 选了 T4 runtime，拿到的不只是一块卡，而是整个 CUDA 平台：



| 层 | 组件 | Colab 里是否自带 |
| :--- | :--- | :--- |
| 编程语言 | CUDA C/C++、Numba、CuPy、PyTorch | 有（Numba / CuPy / PyTorch 预装） |
| 编译器 | `nvcc`、`ptxas`、`cuobjdump` | 有（`/usr/local/cuda/bin`） |
| 开发工具 | Nsight Systems、Nsight Compute、Compute Sanitizer | 部分（`compute-sanitizer` 随 Toolkit 安装） |
| 数学与深度学习库 | cuBLAS、cuDNN、NCCL、TensorRT | 部分（cuBLAS、cuDNN 随 PyTorch 安装） |
| 管理接口 | NVML（`nvidia-smi`、`pynvml`） | 有（[Lab 06](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/)） |
| 硬件特性 | Tensor Core、MPS | T4 支持 Tensor Core 和 MPS；不支持 MIG（需要 Ampere 及更新架构） |



这张表就是本篇其它 Lab 在用的东西：Lab 03 验证的是“编译器”层，Lab 06 用的是“管理接口”层，下一章写的是“编程语言”层，[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 碰的是“硬件特性”层。

## 第 1 步：确认 T4 的 Compute Capability

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!nvcc --version | tail -2

实测输出（版本号以你实际看到的为准）：

```text
name, compute_cap, driver_version
Tesla T4, 7.5, 580.82.07
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
```

第二行的 `7.5` 就是 T4 的 Compute Capability，编译时要传给 nvcc 的 `-arch=sm_75` 就来自它。

Colab 的 T4 runtime **预装了完整 CUDA Toolkit**，`nvcc` 在 `/usr/local/cuda/bin`，不需要 `apt-get` 安装任何东西。

## 第 2 步：一个 block、一个 thread

在 Colab 里用 `%%writefile` 魔法把源码写到文件，再用 `!nvcc` 编译：

In [ ]:
%%writefile hello.cu
#include <cstdio>

__global__ void hello() {
    printf("Hello from block %d, thread %d\n", blockIdx.x, threadIdx.x);
}

int main() {
    hello<<<1, 1>>>();
    cudaDeviceSynchronize();
    return 0;
}

In [ ]:
!nvcc -arch=sm_75 -o hello hello.cu && ./hello

实测输出：

```text
Hello from block 0, thread 0
```

只有一行，因为只有一个线程。两个 ID 都是 0，因为唯一的 block 和唯一的 thread 编号都从 0 开始。

## 第 3 步：2 个 block、4 个 thread

把 `hello<<<1, 1>>>()` 改成 `hello<<<2, 4>>>()`，重新编译运行：

In [ ]:
!sed -i 's/hello<<<1, 1>>>/hello<<<2, 4>>>/' hello.cu && nvcc -arch=sm_75 -o hello hello.cu && ./hello

实测输出 8 行（顺序**不保证**，多跑几次可能不同）：

```text
Hello from block 1, thread 0
Hello from block 1, thread 1
Hello from block 1, thread 2
Hello from block 1, thread 3
Hello from block 0, thread 0
Hello from block 0, thread 1
Hello from block 0, thread 2
Hello from block 0, thread 3
```

注意实测时 block 1 比 block 0 先打印。观察两点：同一个 block 内的线程通常连在一起（它们在同一个 SM、同一个 warp 里），而 block 0 和 block 1 谁先打印是不确定的，因为两个 block 可能落在不同的 SM 上**同时**执行。这就是“GPU 不保证 block 顺序”的第一次亲眼所见，也是后面 [Lab 08](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-08-cuda-kernel/) 里“越界保护”和“必须同步”的根源。

## 第 4 步：看一眼 PTX

In [ ]:
!nvcc -arch=sm_75 -ptx hello.cu -o hello.ptx && grep -E "^\.version|^\.target|\.entry" hello.ptx

实测输出：

```text
.version 9.0
.target sm_75
.visible .entry _Z5hellov()
```

`.version 9.0` 是 PTX 指令集的版本，CUDA 13.0 生成的是 PTX 9.0；`.target sm_75` 就是 Compute Capability 7.5；`_Z5hellov` 是 C++ 名字修饰后的 `hello()`。你刚才写的 kernel 已经变成了 GPU 的虚拟汇编。

## 第 5 步：故意忘记同步

删掉 `cudaDeviceSynchronize()` 那一行再跑：

In [ ]:
!sed 's/    cudaDeviceSynchronize();//' hello.cu > nosync.cu && nvcc -arch=sm_75 -o nosync nosync.cu && ./nosync; echo "exit code: $?"

实测输出：

```text
exit code: 0
```

kernel 里的 `printf` 一行都没有打印，而 exit code 是 0，表示“程序成功结束”。这是 CUDA 新手最常见的“没有报错的 bug”。

原因是 kernel 启动是**异步**的：`hello<<<…>>>()` 只是把任务交给 GPU 就立刻返回，CPU 不会等它跑完。没有同步，`main` 直接 `return 0`，进程退出时 kernel 的输出还没来得及打印：

![有同步时 CPU 在 cudaDeviceSynchronize 处等 GPU 执行完再返回；删掉同步后 CPU 启动 kernel 就直接退出，kernel 的输出来不及打印](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-07-cuda-programming-model/async-launch.svg)

## 验证

完成后你应该能：

- 用一句话说清 CPU 与 GPU 设计目标的区别，以及“为什么 1 个线程的 kernel 比 CPU 慢”；
- 解释 kernel / thread / block / grid 四者的关系，写出全局线程索引公式；
- 说出 `nvcc` 把 `.cu` 拆成哪两部分、PTX 与 SASS 的区别、`-arch=sm_75` 的含义；
- 说出 T4 的 Compute Capability 以及它决定的两件事；
- 列出 Colab T4 runtime 里免费拿到的 CUDA 平台组件。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `nvcc: command not found` | 极少见；runtime 是 CPU 或镜像异常 | 确认 T4 runtime；`!ls /usr/local/cuda/bin` 检查；必要时 `Runtime → Disconnect and delete runtime` 重新分配 |
| `%%writefile` 报错 | 这一行不是 cell 的第一行 | 魔法命令必须在 cell 第一行 |
| 编译通过但没有任何输出 | 忘了 `cudaDeviceSynchronize()` | 见 第 5 步 |
| `nvcc fatal: Unsupported gpu architecture 'compute_75'` | 未来某个放弃 Turing 的 CUDA 大版本 | CUDA 13.0 仍支持 sm_75；若日后报此错，改用 `-arch=native` 或查当前 Toolkit 支持的最低架构 |
| 输出顺序和书上不一样 | 正常，block 执行顺序不保证 | 不用处理，这正是要观察的现象 |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。